[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_enrichment.ipynb)

# DataSage Stage 2: Enrichment GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
!pip install -q unsloth trl datasets wandb pydantic requests
!pip install -q openenv-core

In [ ]:
import os, sys

REPO_URL = "https://github.com/ricalanis/openenv-hackaton.git"  # UPDATE THIS
REPO_DIR = "/content/datasage"

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

In [ ]:
# Imports
import random
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from training.shared.config import BASE_MODEL, HF_MODEL_REPOS, SPACE_URLS, TRAINING_CONFIGS, WANDB_PROJECT
from training.shared.parsers import parse_enrichment_action
from training.shared.rewards import source_relevance_reward, enrichment_json_format_reward
from environments.enrichment.client import EnrichmentEnv
from environments.enrichment.models import EnrichmentAction

ENV_URL = SPACE_URLS["enrichment"]
STAGE_CONFIG = TRAINING_CONFIGS["enrichment"]
print(f"Environment URL: {ENV_URL}"); print(f"Config: {STAGE_CONFIG}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded: {BASE_MODEL}")

In [ ]:
SYSTEM_PROMPT = """\
You are a data enrichment agent. You enrich enterprise datasets by adding \
derived fields, lookups, and computed columns across multiple domains \
(HR, Sales, Project Management, IT Operations).

Analyze the schema and available sources below, then respond with a JSON \
enrichment action:
{"operation": "<op>", "field_name": "<name>", "source": "<source>", "logic": "<logic>", "params": {}}

Available operations:
- add_field: Add a new enrichment field from a known source
- lookup: Look up external reference data
- compute_derived: Compute a derived metric from existing columns
- add_category: Add a categorical classification

Identify the most valuable enrichment to add and act."""

TASK_DESCRIPTIONS = [
    "Enrich this dataset by adding the most valuable derived field.",
    "Add an enrichment that increases analytical coverage the most.",
    "Look at the available sources and add the most impactful one.",
    "This dataset needs enrichment. Choose the best source to add.",
    "Maximize enrichment coverage by adding the most useful field.",
    "Analyze the schema and pick the enrichment with highest value.",
    "Add a derived field that enables the most downstream analysis.",
    "Choose an enrichment source that fills the biggest analytics gap.",
]

In [ ]:
# Dataset: pre-built with real environment observations
random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []
available_sources_list = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    with EnrichmentEnv(base_url=ENV_URL) as client:
        result = client.reset_with_seed(seed=seed)
        obs = result.observation

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs.domain}\n\nSchema:\n{obs.schema_info}\n\n"
            f"Available Enrichment Sources: {', '.join(obs.available_sources)}\n\n"
            f"Possible Enrichments: {', '.join(obs.possible_enrichments)}\n\n"
            f"Data Preview:\n{obs.data_preview}\n\nTask: {task_desc}"
        )},
    ])
    available_sources_list.append(obs.available_sources)
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
    "available_sources": available_sources_list,
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Environment reward function (calls env directly with stored seeds)
def enrichment_env_reward(completions: list[str], **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for text, seed in zip(completions, seeds):
        try:
            action_dict = parse_enrichment_action(text)
            action = EnrichmentAction(
                operation=action_dict.get("operation", "add_field"),
                field_name=action_dict.get("field_name", "unknown"),
                source=action_dict.get("source", ""),
                logic=action_dict.get("logic", ""),
                params=action_dict.get("params", {}),
            )
            with EnrichmentEnv(base_url=ENV_URL) as client:
                client.reset_with_seed(seed=int(seed))
                result = client.step(action)
                rewards.append(float(result.reward or 0.0))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [ ]:
training_args = GRPOConfig(
    output_dir="./outputs/enrichment-grpo",
    learning_rate=STAGE_CONFIG["learning_rate"], num_train_epochs=STAGE_CONFIG["num_train_epochs"],
    per_device_train_batch_size=STAGE_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=STAGE_CONFIG["gradient_accumulation_steps"],
    num_generations=STAGE_CONFIG["num_generations"],
    max_completion_length=STAGE_CONFIG["max_completion_length"],
    max_prompt_length=STAGE_CONFIG["max_prompt_length"],
    logging_steps=1, save_steps=50, bf16=True,
    use_vllm=True, vllm_mode="colocate",
    report_to="wandb", run_name="datasage-enrichment-grpo-v2")
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

In [ ]:
# Create trainer and train
trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[enrichment_env_reward, source_relevance_reward, enrichment_json_format_reward],
)
print("Starting Stage 2 (Enrichment) GRPO training...")
trainer.train()

In [ ]:
# Save and push to Hub
output_dir = "./outputs/enrichment-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

hf_repo = HF_MODEL_REPOS["enrichment"]
trainer.push_to_hub(hf_repo)
print(f"Model pushed to https://huggingface.co/{hf_repo}")